# PR16 · Keep diffusion volumes and gradient directions together

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PP06](../../curriculum/papers/processing.md#pp06).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** Why must each diffusion volume retain the correct gradient direction before a pathway can be inferred?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Format:** 75–100 minutes for this lesson and its small executable lab, followed by the explicitly labeled upstream practice. Run this notebook from a fresh kernel, top to bottom. Core data are synthetic. Software: NumPy, SciPy, Matplotlib; extra imports are stated in code. This is one stage of a longer processing course, not a replacement for supervised research training.

## Learning objectives

- Explain b-values, b-vectors and b0 images with units.
- Maintain the volume-to-gradient correspondence.
- Rotate directions consistently with a stated coordinate rotation.

## Understand the operation

Diffusion-weighted MRI measures signal attenuation under diffusion-encoding gradients. Each volume is associated with a b-value describing weighting and a direction vector. This fourth dimension is not ordinarily interpreted as a regularly sampled BOLD time course. A b0 image has negligible diffusion weighting and can provide a reference signal. Multiple shells contain groups of similar nonzero b-values, allowing models with different requirements.

B-values commonly use s/mm², directions are dimensionless unit vectors for diffusion-weighted measurements, and diffusivity commonly uses mm²/s. Their product is dimensionless in the exponential signal model. A direction vector also belongs to a coordinate convention. FSL-style files may store3×N values while an analysis API expectsN×3; an ambiguous three-volume file needs metadata rather than shape guessing.

If an image is rotated relative to a fixed frame, gradient directions and/or the fitted tensor must be expressed consistently. Translation alone does not rotate a vector. The exact transformation depends on whether the mapping is active or passive and on file conventions. Never apply an arbitrary affine including shear to b-vectors and renormalize as a universal correction. Diffusion pipelines estimate the relevant motion rotations and produce corresponding rotated gradient tables.

Here an explicit active90-degree rotation maps column vectors byR. We rotate both directions and a tensor asRDRᵀ, verifying unchanged predicted measurements. Rotating only the gradients violates that contract. Separately permuting volumes while retaining the original table also changes the fit, even though all counts and vector norms pass.

## Transformation contract

**Input:** N diffusion measurements, N b-values, N directions and coordinate convention. **Output:** consistent gradients and tensor coordinates. **Preserved under matched rotation:** modeled attenuation. **Lost by unmatched reordering:** measurement identity; a shape check cannot recover it.


## Read the actual course material

- [Oxford FSL: FDT pipeline, TOPUP, EDDY, DTIFIT and TBSS](https://pages.fmrib.ox.ac.uk/fslcourse/practicals/fdt1/index.html). Public university practical; educational data terms at source; linked only.
- [DIPY: diffusion tensor reconstruction](https://github.com/dipy/dipy/blob/05df74a36c48e38ef1aa7420440abb9f1d14e6d4/doc/examples/reconst_dti.py). BSD-3-Clause project; linked source, no example code copied.

Read the named topic alongside this lesson; compare its real-image assumptions with our controlled example. These notebooks use original explanations and original code, not copied upstream passages. The source chapter is the place to continue to a complete real-tool practical. External software and downloaded datasets are not silently run by this notebook.


## Predict, then ask your AI assistant

Use Goose with your installed Ollama model, or ChatGPT. The model is a tutor and code author; the local Python runtime performs these calculations. Paste:

> Audit b-value units, direction norms, table orientation and volume count. State the active-rotation convention before transforming directions. Verify attenuation invariance when both tensor and directions are expressed in the new frame; include the mismatched counterexample. Return at most 20 executable lines per cell, show units and array shapes, and preserve the original. Explain the prediction before running. If an assertion fails, diagnose the disagreement rather than deleting the check.

Write your prediction before executing the reference cells below.


In [1]:
import numpy as np
rng=np.random.default_rng(1616)
g=rng.normal(size=(30,3)); g/=np.linalg.norm(g,axis=1,keepdims=True)
b=np.full(30,1000.); D=np.diag([.0017,.0003,.0003])
R=np.array([[0.,-1,0],[1,0,0],[0,0,1]])
rotated_g=g@R.T; rotated_D=R@D@R.T
atten=lambda directions,tensor: np.exp(-b*np.einsum('ni,ij,nj->n',directions,tensor,directions))
original=atten(g,D); consistent=atten(rotated_g,rotated_D)
wrong=atten(rotated_g,D)
assert np.allclose(original,consistent) and not np.allclose(original,wrong)
assert np.allclose(np.linalg.norm(g,axis=1),1)
print('matched rotation error:',np.max(abs(original-consistent)))
print('unmatched rotation error:',np.mean(abs(original-wrong)))


matched rotation error: 1.1102230246251565e-16
unmatched rotation error: 0.2191665808907843


In [2]:
bvals=np.r_[0,b]; bvecs=np.vstack([np.zeros(3),g]); signal=np.r_[1,original]
assert len(signal)==len(bvals)==len(bvecs)
assert np.allclose(bvecs[bvals>0].sum(1)*0+np.linalg.norm(bvecs[bvals>0],axis=1),1)
permuted_signal=signal.copy(); permuted_signal[1:]=signal[:0:-1]
print('wrong volume/table matching MSE:',np.mean((permuted_signal-signal)**2))
assert np.mean((permuted_signal-signal)**2)>.01
print('b0 row has zero direction by this convention; do not normalize it to unit length.')


wrong volume/table matching MSE: 0.06285921343100756
b0 row has zero direction by this convention; do not normalize it to unit length.


## Check and explain

All diffusion-weighted directions have unit norm and the b0 direction is zero by the stated convention. Consistent rotation preserves every signal; unmatched rotation or reversed volume ordering does not. Norms and dimensions alone cannot catch either mismatch.

## Deliberately wrong method

Normalizing a b0 zero vector produces undefined values. Sorting the image volumes without identically sorting the gradient table breaks their identity. Applying the wrong rotation convention can leave plausible scalar maps while corrupting orientation.

## Transfer to an actual dataset or tool — guided assignment

Complete the diffusion-data inspection section of Oxford FDT and the DIPY tensor input chapter. Inspect actual bval/bvec files alongside a DWI volume series, identify b0 and shells, and record the image/gradient coordinate convention. For an eddy-corrected derivative, identify the rotated bvec output and verify it is the table used downstream. Do not manually rotate twice because an assistant notices the word “motion” in two filenames.

**Submit:** a transformation card, one labeled figure or numerical result, the failed-method diagnosis, and the upstream-practice evidence. If the external exercise has not been run, mark it **not executed** and state the missing software/data; do not convert a proposed command into a claimed result.

## Exit questions and answer key

1. Why is a diffusion fourth dimension not automatically time? **Answer:** it indexes acquisitions with different diffusion encodings.
2. Why rotate directions? **Answer:** the direction of encoding relative to anatomy changes under coordinate/pose changes; the signal model must use consistent coordinates.


### Return to the research question

Reopen [PP06](../../curriculum/papers/processing.md#pp06) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
